# Le théorème central limite
*A partir du cours de deeplearning AI*


Le théorème central limite établit que la distribution de la **moyenne** d'un grand nombre de variables aléatoires IID (indépendantes et identiquement distribuées) converge vers une distribution normale, quelle que soit la distribution d'origine des variables.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm

# --- Fonctions utilistaires ---
def plot_kde_and_qq(moyennes, mu, sigma):

    x_range = np.linspace(min(moyennes), max(moyennes), 100)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # --- Graphique gauche : histogramme + KDE + gaussienne théorique ---
    axes[0].hist(moyennes, bins=50, density=True, color="steelblue", alpha=0.5, label="hist")

    from scipy.stats import gaussian_kde
    kde = gaussian_kde(moyennes)
    axes[0].plot(x_range, kde(x_range), color="crimson", linestyle="dashed", label="kde")

    axes[0].plot(x_range, norm.pdf(x_range, loc=mu, scale=sigma), color="black", label="gaussienne théorique")

    axes[0].legend()
    axes[0].set_title("KDE vs Gaussienne théorique")

    # --- Graphique droit : QQ-plot ---
    stats.probplot(moyennes, plot=axes[1], fit=True)
    axes[1].set_title("QQ-plot")

    plt.tight_layout()
    plt.show()

## 1. Population gaussienne

La population utilisée dans cette partie suit une distribution gaussienne, générée avec `np.random.normal`. On fixe μ=10 et σ=5 pour un total de 100 000 observations — une taille suffisamment grande pour considérer qu'on travaille sur une population, pas un échantillon.

In [ ]:
mu=10
sigma=5
population_gaussienne=np.random.normal(mu, sigma, 100000)
plt.hist(population_gaussienne,bins=100, density=True,color="blue") #usage de density=True pour avoir la densité et non le comptage brut
plt.title("Distribution gaussienne");

### 1.1 Échantillonnage de la population

Dans ce TP, les données sont simulées : nous avons fixé μ=10 et σ=5 avant de générer la population. On pourrait donc calculer la moyenne et l'écart type directement sur les 100 000 observations et retrouver des valeurs proches des paramètres théoriques. C'est exactement ce que fait la cellule ci-dessous.

Mais ce n'est pas le cas réel. En pratique, on n'a jamais accès à la population entière — seulement à un échantillon. L'enjeu de la statistique inférentielle est précisément d'estimer les paramètres de la population (μ, σ) à partir de cet échantillon partiel.

C'est là qu'intervient le **Théorème Central Limite (TCL)** : si on tire un grand nombre d'échantillons d'une population et qu'on calcule la moyenne de chaque échantillon, la distribution de ces moyennes tend vers une loi normale — quelle que soit la distribution de la population de départ. Cette propriété est ce qui rend l'inférence statistique possible.

Pour voir le TCL en action, il faut donc une fonction capable de :
1. Tirer des échantillons aléatoires de la population (avec remise)
2. Calculer la moyenne de chaque échantillon
3. Renvoyer l'ensemble de ces moyennes sous forme de tableau NumPy

C'est le rôle de la fonction `moyenne_echantillon` définie ci-dessous.

> **Avec remise** signifie qu'un même individu peut être sélectionné plusieurs fois dans le même échantillon. Cela garantit l'indépendance entre les tirages, condition nécessaire pour que le TCL s'applique.

Le théorème stipule que si la valeur de la taille de l'échantillon est suffisamment grande, la distribution des moyennes d'échantillons devrait être gaussienne. C'est ce que la suite permet de constater.

In [ ]:
moyenne_population_G=np.mean(population_gaussienne)
ecarttype_population_G=np.std(population_gaussienne)
print(f"La population gaussienne a pour moyenne {moyenne_population_G:.1f} et pour écart type {ecarttype_population_G:.1f}")

In [ ]:
def moyenne_echantillon(data, taille_echantillon):
    # Conservez tous les moyennes dans une liste
    moyenne = []

    # Pour un grand nombre d'échantillons
    # Cette valeur n'a pas d'incidence sur le théoreme, mais sur la qualité des histogrammes
    for _ in range(10_000):
        # Obtenir un échantillon des données AVEC remplacement
        echantillon = np.random.choice(data, size=taille_echantillon)

        # Sauvegarde la moyenne de l'echantillon
        moyenne.append(np.mean(echantillon))


    return np.array(moyenne)

In [ ]:
# Calculer les moyennes d'échantillon
moyenne_echantillon_g = moyenne_echantillon(population_gaussienne, taille_echantillon=5)

# Dessine un histogramme avec les valeurs moyennes
plt.hist(moyenne_echantillon_g,bins=50, density=True,color="pink")
plt.title("Distribution des moyennes d'échantillon");

### 1.2 Vérification visuelle

La distribution des moyennes d'échantillon a l'air gaussienne — mais ça ne prouve rien. Avec une taille d'échantillon aussi petite (n=5), c'est peut-être juste de la chance visuelle.

Pour vérifier sérieusement, on calcule ce que la distribution *devrait* être si le TCL est valide. La théorie prédit que les moyennes d'échantillon suivent une loi normale dont les paramètres se déduisent directement de la population :

$$\mu_{\bar{X}} = \mu \qquad \sigma_{\bar{X}} = \frac{\sigma}{\sqrt{n}}$$

où $n$ est la taille de chaque échantillon. La moyenne des moyennes reste égale à la moyenne de la population. L'écart-type, lui, se réduit : plus les échantillons sont grands, plus les moyennes se resserrent autour de μ. C'est le **resserrement par la racine carrée** — une propriété fondamentale qu'on retrouvera partout en ML.

On trace cette courbe gaussienne théorique par-dessus l'histogramme des moyennes simulées. Si les deux se superposent, le TCL tient.


In [ ]:
mu_moyenne_echantillon=mu
sigma_moyenne_echantillon=sigma/np.sqrt(5)

#Définir l'intervalle des abscisses (x_range) pour la courbe gaussienne pour le tracé
x_range=np.linspace(min(moyenne_echantillon_g),max(moyenne_echantillon_g),100)

plt.hist(moyenne_echantillon_g, bins=50, density=True, color="pink")
plt.plot(
    x_range,
    norm.pdf(x_range, loc=mu_moyenne_echantillon, scale=sigma_moyenne_echantillon),
    color="red",
)
plt.show()

### 1.3 Estimation de densité par noyau (KDE)

On peut aller plus loin avec une **estimation de densité par noyau (KDE)** : une courbe lisse qui estime la PDF des moyennes d'échantillon sans supposer de forme particulière. Si cette courbe ressemble à la gaussienne théorique, la convergence est confirmée.



In [ ]:
plt.hist(moyenne_echantillon_g, bins=50, density=True, color="pink", label="hist")

# va estimer la PDF de la moyenne des échantillons
sns.kdeplot(
    data=moyenne_echantillon_g,
    color="grey",
    label="kde",
    linestyle="dashed",
    fill=True,
)

plt.plot(
    x_range,
    norm.pdf(x_range, loc=mu_moyenne_echantillon, scale=sigma_moyenne_echantillon),
    color="steelblue",
    label="gaussian",
)
plt.legend()
plt.show()

### 1.4 QQ-plot

Une troisième méthode de vérification : le **graphique quantile-quantile (QQ-plot)**. Il compare les quantiles empiriques des moyennes d'échantillon aux quantiles théoriques d'une loi normale. Si les points s'alignent sur une droite, la distribution est gaussienne. Toute déviation — notamment aux extrémités — signale que les queues de distribution s'écartent de la normale.

Le résultat ici est une droite quasi parfaite, ce qui confirme que les moyennes d'échantillon suivent bien une distribution gaussienne — même avec n=5, propriété spécifique à la distribution gaussienne elle-même.



In [ ]:
fig, ax=plt.subplots(figsize=(6,6))
res=stats.probplot(moyenne_echantillon_g, plot=ax, fit=True)
plt.show()

## 2. Population binomiale

La population utilisée dans cette partie suit une distribution binomiale, générée avec ``np.random.binomial``. On fixe n=5 (nombre d'essai) et p=0,8 (probabilité de succes) pour un total de 100 000 observations.

In [ ]:
n=5
p=0.8

population_binomiale=np.random.binomial(n,p,100000)
sns.histplot(population_binomiale, stat="count")
plt.show()

### 2.1 Paramètres de la distribution binomiale

Contrairement au cas gaussien, μ et σ ne sont pas des paramètres directs de la binomiale — on ne les a pas fixés explicitement pour générer les données. On peut les estimer en calculant mean et std sur la population entière, mais en pratique cette population n'est pas accessible.

Heureusement, les paramètres de la binomiale sont analytiquement connus :

$$\mu = np \qquad \sigma = \sqrt{np(1-p)}$$

Ce qui permet de les calculer sans toucher aux données — utile dès qu'on travaille sur un échantillon réel.

In [ ]:
moyenne_population_b=np.mean(population_binomiale)
ecarttype_population_b=np.std(population_binomiale)
print(f"La moyenne de la population binomiale est {moyenne_population_b:.1f} et son écart type est {ecarttype_population_b:.1f} ")

In [ ]:
moyenne_population_b=n*p
ecarttype_population_b=np.sqrt(n*p*(1-p))
print(f"La moyenne de l'echantillon binomiale est {moyenne_population_b:.1f} et son écart type est {ecarttype_population_b:.1f} ")

### 2.2 Condition d'applicabilité du TCL — règle empirique

Avant de vérifier le TCL sur la binomiale, il faut savoir qu'il ne tient pas automatiquement. La règle empirique standard est :

$$\min(Np,\ N(1-p)) \geq 5 \quad \text{avec } N = n \times \text{taille\echantillon}$$

Si cette condition est satisfaite, le TCL devrait tenir. Dans le cas contraire, les moyennes d'échantillon ne convergent pas vers une gaussienne, quelle que soit la taille des échantillons.

> Cette règle reste une approximation. D'autres facteurs entrent en jeu : présence d'outliers, objectif de l'analyse, asymétrie de la distribution. Ne pas l'appliquer mécaniquement.


#### 2.2.1 Vérification avec petit échantillon (taille=3)
Avec `taille_echantillon=3`, on obtient `N = 5 * 3 = 15`, soit `min(12, 3) = 3` — la condition n'est pas satisfaite. On s'attend donc à ce que le TCL ne tienne pas.

On calcule les paramètres théoriques des moyennes d'échantillon :

$$\mu_{\bar{X}} = \mu \qquad \sigma_{\bar{X}} = \frac{\sigma}{\sqrt{\text{taille\echantillon}}}$$

La visualisation KDE vs courbe gaussienne et le QQ-plot confirment : avec un échantillon aussi petit, les moyennes ne suivent pas une distribution gaussienne.

In [ ]:
taille_echantillon_b=3
N=n*taille_echantillon_b

valeur_condition=np.min([N*p,N*(1-p)])
print(f"La valeur conditionnelle est : {int(valeur_condition*10)/10:.1f}. La condition est-elle remplie? : {valeur_condition>=5}")

In [ ]:
#Utilisation de la fonction définie plus tôt moyenne_echantillon, elle st générique et s'applique à n'importe quelle distribution
moyenne_echantillon_b=moyenne_echantillon(population_binomiale, taille_echantillon=taille_echantillon_b)
mu_moyenne_echantillon_b=n*p
sigma_moyenne_echantillon_b=np.sqrt(n*p*(1-p))/np.sqrt(taille_echantillon_b)

In [ ]:
plot_kde_and_qq(moyenne_echantillon_b, mu_moyenne_echantillon_b,sigma_moyenne_echantillon_b)

#### 2.2.2 Vérification avec grand échantillon (taille=30)

Avec `taille_echantillon=30`, on obtient `N = 5 * 30 = 150`, soit `min(120, 30) = 30` — la condition est satisfaite. Le TCL devrait tenir.

Les paramètres théoriques restent les mêmes formules, appliquées avec la nouvelle taille d'échantillon. Cette fois, la KDE et le QQ-plot s'alignent sur la gaussienne théorique : le TCL tient.

> **Ce que montre cette comparaison :** la convergence vers la normale n'est pas immédiate sur une distribution asymétrique. Il faut suffisamment d'observations pour que la moyenne "lisse" l'asymétrie de départ. C'est précisément ce que quantifie la condition d'applicabilité.

In [ ]:
taille_echantillon_b=30
N=n*taille_echantillon_b
moyenne_echantillon_b=moyenne_echantillon(population_binomiale, taille_echantillon=taille_echantillon_b)
mu_moyenne_echantillon_b=n*p
sigma_moyenne_echantillon_b=np.sqrt(n*p*(1-p))/np.sqrt(taille_echantillon_b)
plot_kde_and_qq(moyenne_echantillon_b, mu_moyenne_echantillon_b,sigma_moyenne_echantillon_b)

## 3. Distribution de Poisson

La distribution de Poisson modélise le nombre d'événements survenant dans un intervalle de temps ou d'espace fixe, étant donné un taux moyen d'occurrence μ. Ses paramètres sont :

$$\mu = \mu \qquad \sigma = \sqrt{\mu}$$

Le processus de vérification étant identique aux sections précédentes, on passe directement à la visualisation avec `taille_echantillon=30`. Le TCL devrait tenir — la distribution de Poisson satisfait les conditions nécessaires pour des échantillons suffisamment grands.

In [ ]:
# Paramètres de la Poisson : mu = mu, sigma = sqrt(mu)
mu_poisson = 3  # choisis la valeur que tu veux

population_poisson = np.random.poisson(mu_poisson, 100_000)

taille_echantillon_p = 30
moyenne_echantillon_p = moyenne_echantillon(population_poisson, taille_echantillon=taille_echantillon_p)

mu_moyenne_echantillon_p = mu_poisson
sigma_moyenne_echantillon_p = np.sqrt(mu_poisson) / np.sqrt(taille_echantillon_p)

plot_kde_and_qq(moyenne_echantillon_p, mu_moyenne_echantillon_p, sigma_moyenne_echantillon_p)

## 4. Distribution de Cauchy

La distribution de Cauchy est un cas limite : elle possède des **queues lourdes**, ce qui signifie que les valeurs extrêmes sont bien plus fréquentes que dans une gaussienne de même dispersion. Surtout, elle n'a **ni moyenne ni variance définies** au sens mathématique — les intégrales qui les définissent divergent.

C'est précisément pourquoi le TCL ne s'applique pas : il requiert que la variance de la distribution soit finie. Sans variance finie, la moyenne d'échantillon ne converge vers rien de stable.

On le vérifie avec `taille_echantillon=30` d'abord — seuil habituellement suffisant — puis avec `taille_echantillon=100`. Dans les deux cas, le QQ-plot s'écarte fortement d'une droite : les queues dévient massivement, signature d'une distribution non gaussienne.

> **À retenir :** le TCL est puissant mais pas universel. Avant de l'invoquer, vérifier que la distribution sous-jacente a une variance finie. La Cauchy est l'exemple canonique de ce que le TCL ne peut pas dompter.

In [ ]:
cauchy_population=np.random.standard_cauchy(1000)
sns.histplot(cauchy_population,stat="density",label="hist")
plt.show()

**Avec une taille d'échantillon n=30**

In [ ]:
moyenne_echantillon_c=moyenne_echantillon(cauchy_population,taille_echantillon=30)
fig, ax=plt.subplots(figsize=(6,6))
res=stats.probplot(moyenne_echantillon_c, plot=ax, fit=True)
ax.set_title("QQ-plot - Taille échantillon = 30")
plt.show()

**Avec une taille d'échantillon n=100**

In [ ]:
moyenne_echantillon_c=moyenne_echantillon(cauchy_population,taille_echantillon=100)
fig, ax=plt.subplots(figsize=(6,6))
res=stats.probplot(moyenne_echantillon_c, plot=ax, fit=True)
plt.show()